# Quickstart

Load the frozen encoder, describe an array, and extract per-contact features.

The encoder reads two things about the array: which array each contact sits on with its number
along that array, and which anatomical region it falls in. Both come from the sidecar. The encoder weights remain frozen during feature extraction.

For raw recordings, start with `mapa.preprocessing.prepare_recording`; see
[the preprocessing guide](../docs/PREPROCESSING.md) or the complete runnable
[voltage-to-readout example](../examples/decode_recording.py). This notebook focuses on the lower-level band API.


In [ ]:
import torch

import mapa as sp

CKPT = None   # download the released weights, or set a local checkpoint path
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Describe the array

`build_sidecar` parses the clinical electrode labels into an array name and a contact number.
Region IDs index `V14_DKT_REGION_LABELS` in `mapa/data/anatomy.py`: 0–73 for atlas regions
and 74 for the reserved entry. The vocabulary distinguishes hemispheres. Keep these IDs fixed
across recordings; the random IDs below are only for this synthetic example.

In [ ]:
labels = [f"{array}{i}" for array in ("LA", "LH", "RO") for i in range(1, 9)]
n_contacts = len(labels)

torch.manual_seed(0)
region_ids = torch.randint(0, 74, (n_contacts,))   # replace with your own atlas lookup

sidecar = sp.build_sidecar(labels, region_id=region_ids)
print(f"{n_contacts} contacts on {sidecar.n_arrays} arrays")

## 2. Load the encoder

Load the released model, with its architecture and spatial encodings read from the checkpoint.
The encoder weights are frozen. With `CKPT = None`, the hub entry point downloads the checkpoint
and caches it locally. Set `MAPA_WEIGHTS` to a local checkpoint directory to work offline.

In [ ]:
if CKPT is not None:
    encoder = sp.MapaEncoder.from_checkpoint(CKPT, device=DEVICE)
else:
    from mapa.hub.backbones import mapa_vits384
    encoder = mapa_vits384(device=DEVICE)

n_params = sum(p.numel() for p in encoder.tower.parameters())
print(f"width {encoder.d_model}, {n_params / 1e6:.1f} M parameters")

## 3. Prepare the session

The token layout depends on the array and the window length, not on the window itself, so build
it once per recording session and reuse it for every batch.

In [ ]:
N_TIME = 32   # frames on the 32 Hz clock, so one second
session = encoder.prepare(sidecar, n_time=N_TIME)
print(f"{session.k_full} tokens per contact per window")

## 4. Encode

The inputs are three normalized magnitude STFT bands, each shaped
`(batch, contacts, bins, frames)`. All three arrive on the shared 32 Hz frame clock, and the
frontend decimates them to 4 Hz, 16 Hz and 32 Hz. Real inputs must follow the paper's
contact selection, referencing, STFT, and session normalization. Neuroprobe uses the included
frozen Guard 1 exclusions before referencing; Guard 2 does not reject evaluation windows.
The encoder applies Guard 3 clipping to normalized bands, including tap 0. See the
[evaluation guide](../evals/README.md) for the contact-selection procedure.

In [ ]:
batch = 4
bands = [torch.randn(batch, n_contacts, bins, N_TIME, device=DEVICE)
         for bins in (7, 6, 7)]

features = encoder(bands, session, taps=(0, 3, 6, 9, 12))
for tap, value in features.items():
    print(f"tap {tap:2d}  {tuple(value.shape)}")

Tap 0 returns the frontend baseline input features: `(batch, contacts, 1, 348)` for one second.
These features are identical across ablation checkpoints. Encoder taps return
`(batch, contacts, 52, 384)` before the output LayerNorm.

Contacts are grouped by array in `session.contact_order`. To restore the original input order,
use `features[12][:, session.contact_order.argsort()]`. Keep the grouped order for the pooling
example below.

## 5. Summarize for a readout

Within a subject, keep the per-contact features. For cross-subject decoding, average contacts
within each anatomical region.

In [ ]:
head = features[12]
flat = head.reshape(head.shape[0], head.shape[1], -1)
print(f"per contact  {tuple(flat.shape)}")

pooled = sp.pool_to_regions(flat, session)
print(f"per region   {tuple(pooled.shape)}")

`pooled` contains one feature vector per region present in this session, in sorted region-ID
order. The IDs are `session.region_of_contact.unique()`. Align shared region IDs across sessions
before fitting a cross-subject readout. See `evals/README.md` for the published evaluation.